In [1]:
!pip install fastapi uvicorn langchain langchain-community langchain-openai faiss-cpu pydantic pypdf nest-asyncio


[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip install --upgrade python-multipart


[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Set Your OpenAI API Key

In [1]:
from getpass import getpass
import os

# This will trigger the secure input box at the top of your VS Code window
OPENAI_API_KEY = getpass("Enter your new OpenAI API key: ").strip()

if not OPENAI_API_KEY:
  raise ValueError("API key cannot be empty.")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print("API key configured successfully for this notebook session.")

API key configured successfully for this notebook session.


Run the FastAPI Application Code

In [2]:
from typing import List
from fastapi import FastAPI, File, HTTPException, UploadFile, status
from pydantic import BaseModel
import nest_asyncio
import os
import uvicorn
import traceback

# LangChain Imports
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Enable async loop inside Jupyter
nest_asyncio.apply()

app = FastAPI(
    title="RAG Document Processing API",
    description="Upload documents (PDF, TXT), chunk text with metadata, and query via LLM.",
    version="1.0.0",
    redirect_slashes=True,  # Handles both /query and /query/ cleanly
)

# Configuration Setup
FAISS_INDEX_PATH = "faiss_index"
UPLOAD_DIR = "uploaded_docs"
os.makedirs(UPLOAD_DIR, exist_ok=True)
os.makedirs(FAISS_INDEX_PATH, exist_ok=True)

# Pull the API key dynamically from environment memory
ACTIVE_API_KEY = os.getenv("OPENAI_API_KEY")

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small", openai_api_key=ACTIVE_API_KEY
)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)


class QueryRequest(BaseModel):
  question: str
  top_k: int = 4


class QueryResponse(BaseModel):
  answer: str
  sources: List[str]


@app.post("/upload", status_code=status.HTTP_201_CREATED)
async def upload_document(file: UploadFile = File(...)):
  if not file.filename:
    raise HTTPException(status_code=400, detail="Invalid filename or empty upload.")

  file_ext = os.path.splitext(file.filename)[1].lower()
  file_path = os.path.join(UPLOAD_DIR, file.filename)

  try:
    with open(file_path, "wb") as buffer:
      content = await file.read()
      if len(content) == 0:
        raise HTTPException(status_code=400, detail="Uploaded file is empty (0 bytes).")
      buffer.write(content)
  except Exception as e:
    raise HTTPException(status_code=500, detail=f"Failed to save file: {str(e)}")

  try:
    if file_ext == ".pdf":
      loader = PyPDFLoader(file_path)
      docs = loader.load()
    elif file_ext == ".txt":
      loader = TextLoader(file_path, encoding="utf-8")
      docs = loader.load()
    else:
      if os.path.exists(file_path):
        os.remove(file_path)
      raise HTTPException(
          status_code=400,
          detail="Unsupported file format. Only .pdf and .txt files are allowed.",
      )

    for doc in docs:
      doc.metadata.update({"source_file": file.filename, "file_type": file_ext})

    chunks = text_splitter.split_documents(docs)
    if not chunks:
      raise HTTPException(status_code=400, detail="Document yielded no text chunks.")

    if os.path.exists(os.path.join(FAISS_INDEX_PATH, "index.faiss")):
      vector_store = FAISS.load_local(
          FAISS_INDEX_PATH, embeddings, allow_dangerous_deserialization=True
      )
      vector_store.add_documents(chunks)
    else:
      vector_store = FAISS.from_documents(chunks, embeddings)

    vector_store.save_local(FAISS_INDEX_PATH)

    return {
        "filename": file.filename,
        "status": "Success",
        "chunks_indexed": len(chunks),
        "message": "Document processed, chunked, and indexed successfully.",
    }

  except Exception as e:
    if isinstance(e, HTTPException):
      raise e
    # This prints the precise traceback to your Jupyter notebook console for debugging 500 errors
    traceback.print_exc()
    raise HTTPException(status_code=500, detail=f"Document processing error: {str(e)}")


@app.post("/query", response_model=QueryResponse)
async def query_rag(request: QueryRequest):
  if not request.question.strip():
    raise HTTPException(status_code=400, detail="Query string cannot be empty.")

  if not os.path.exists(os.path.join(FAISS_INDEX_PATH, "index.faiss")):
    raise HTTPException(
        status_code=404,
        detail="No vector index found. Please upload documents via /upload first.",
    )

  try:
    vector_store = FAISS.load_local(
        FAISS_INDEX_PATH, embeddings, allow_dangerous_deserialization=True
    )
    retriever = vector_store.as_retriever(search_kwargs={"k": request.top_k})
    retrieved_docs = retriever.invoke(request.question)

    context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    sources = list(
        set([doc.metadata.get("source_file", "Unknown") for doc in retrieved_docs])
    )

    llm = ChatOpenAI(
        model="gpt-4o-mini", temperature=0.2, openai_api_key=ACTIVE_API_KEY
    )
    prompt_template = ChatPromptTemplate.from_messages([
        (
            "system",
            (
                "You are a precise technical assistant. Answer the user's"
                " question strictly using only the provided context elements."
                " If the answer cannot be found in the context, state 'I cannot"
                " find the answer in the provided documents.'"
            ),
        ),
        ("human", "Context:\n{context}\n\nQuestion: {question}"),
    ])

    chain = prompt_template | llm
    response = chain.invoke({"context": context, "question": request.question})

    return QueryResponse(answer=response.content, sources=sources)

  except Exception as e:
    traceback.print_exc()
    raise HTTPException(status_code=500, detail=f"Error processing query: {str(e)}")


@app.get("/health", status_code=status.HTTP_200_OK)
def health_check():
  return {"status": "healthy", "service": "RAG API running"}

C:\Users\prati\AppData\Local\Temp\ipykernel_8704\1418083362.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, TextLoader


Start the Server

In [ ]:
import threading
import uvicorn


def run_server():
  uvicorn.run(app, host="127.0.0.1", port=8001, log_level="info")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
print("FastAPI server started! Go to http://127.0.0.1:8001/docs")

FastAPI server started! Go to http://127.0.0.1:8001/docs


INFO:     Started server process [8704]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8001 (Press CTRL+C to quit)


INFO:     127.0.0.1:51181 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:51181 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:55383 - "POST /upload HTTP/1.1" 500 Internal Server Error


Traceback (most recent call last):
  File "C:\Users\prati\AppData\Local\Temp\ipykernel_8704\1418083362.py", line 96, in upload_document
    vector_store = FAISS.from_documents(chunks, embeddings)
  File "c:\Users\prati\AppData\Local\Programs\Python\Python313\Lib\site-packages\langchain_core\vectorstores\base.py", line 814, in from_documents
    return cls.from_texts(texts, embedding, metadatas=metadatas, **kwargs)
           ~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\prati\AppData\Local\Programs\Python\Python313\Lib\site-packages\langchain_community\vectorstores\faiss.py", line 1043, in from_texts
    embeddings = embedding.embed_documents(texts)
  File "c:\Users\prati\AppData\Local\Programs\Python\Python313\Lib\site-packages\langchain_openai\embeddings\base.py", line 757, in embed_documents
    return self._get_len_safe_embeddings(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        texts, engine=engine, chunk_size=chunk_size, **kwargs
        ^^^^^

INFO:     127.0.0.1:59690 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:59690 - "GET /openapi.json HTTP/1.1" 200 OK


In [6]:
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import seaborn as sns

# Initialize LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, openai_api_key=ACTIVE_API_KEY)


# Define Custom Tools
@tool
def calculate_summary_statistics(file_path: str) -> str:
  """Loads a CSV file and calculates basic descriptive statistics and missing values."""
  try:
    df = pd.read_csv(file_path)
    buffer = [
        f"Dataset Shape: {df.shape}",
        f"Columns & Types:\n{df.dtypes.to_string()}",
        f"Missing Values:\n{df.isnull().sum().to_string()}",
        f"Summary Statistics:\n{df.describe().to_string()}",
    ]
    return "\n\n".join(buffer)
  except Exception as e:
    return f"Error processing file: {str(e)}"


@tool
def generate_correlation_heatmap(
    file_path: str, save_path: str = "uploaded_docs/heatmap.png"
) -> str:
  """Generates a Seaborn correlation heatmap and saves it as an image file."""
  try:
    df = pd.read_csv(file_path)
    numeric_df = df.select_dtypes(include=[np.number])

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        numeric_df.corr(), annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5
    )
    plt.title("Correlation Heatmap")
    plt.tight_layout()

    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path)
    plt.close()
    return f"Successfully generated and saved correlation heatmap to {save_path}"
  except Exception as e:
    return f"Error generating heatmap: {str(e)}"


tools = [calculate_summary_statistics, generate_correlation_heatmap]

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        (
            "You are an expert Data Analyst AI Agent. Use your available tools"
            " to analyze datasets and generate charts."
        ),
    ),
    ("placeholder", "{chat_history}"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

print("Data Analyst Agent is fully configured and ready!")

ImportError: cannot import name 'AgentExecutor' from 'langchain.agents' (c:\Users\prati\AppData\Local\Programs\Python\Python313\Lib\site-packages\langchain\agents\__init__.py)